# Plot variables

In [ ]:
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Load data

In [ ]:
fi = {}
lc = {}
perf = []

for reduction in ["fa"]:
    fi[reduction] = {}
    lc[reduction] = {}
    for model_type in ["lr", "rf"]:
        fi[reduction][model_type] = {}
        lc[reduction][model_type] = {}
        for cluster in [1, 3, 4, 5, 6, 7, 8]:
            with open(
                f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib",
                "rb",
            ) as f:
                model = joblib.load(f)
                if model_type == "rf":
                    fi[reduction][model_type][cluster] = model.feature_importances_
                else:
                    lc[reduction][model_type][cluster] = model.local_coef_
                perf.append(
                    pd.Series(
                        {
                            "reduction": reduction,
                            "model": model_type,
                            "cluster": cluster,
                            "accuracy": model.score_,
                            "balanced_accuracy": model.balanced_accuracy_,
                            "precision": model.precision_,
                            "recall": model.recall_,
                            "f1_macro": model.f1_macro_,
                            "f1_micro_": model.f1_micro_,
                            "f1_weighted": model.f1_weighted_,
                        }
                    )
                )

In [ ]:
census = gpd.read_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

selection = [
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Zaměstnaní - Specialisté",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba",
    "Počet obyvatel na dům",
    "Obyvatelstvo - věk: 7 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 24  - celkem",
    "Obyvatelstvo - věk: 45 - 54  - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby v domácnosti, děti předškolního věku, ostatní závislé osoby - celkem",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - s dlouhodobým pobytem - celkem",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem",
    "geometry",
]

fas = census[selection]

In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = fas.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(["geometry", "kod_nadzsj_d", "final_without_noise"])

data["Cluster"] = data["final_without_noise"].map(cluster_mapping[3])

## RF

In [ ]:
models = []

for reduction in ["fa"]:
    for model_type in ["rf"]:
        for cluster in [1, 3, 4, 5, 6, 7, 8]:
            path = f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib"
            with open(path, "rb") as f:
                model = joblib.load(f)

            models.append(
                {
                    "reduction": reduction,
                    "model_type": model_type,
                    "cluster": cluster,
                    "model": model,
                }
            )

df_models = pd.DataFrame(models)

In [ ]:
morpho_names = {
    1: "Incoherent Large-Scale Homogeneous Fabric",
    2: "Incoherent Large-Scale Heterogeneous Fabric",
    3: "Incoherent Small-Scale Linear Fabric",
    4: "Incoherent Small-Scale Sparse Fabric",
    5: "Incoherent Small-Scale Compact Fabric",
    6: "Coherent Interconnected Fabric",
    7: "Coherent Dense Disjoint Fabric",
    8: "Coherent Dense Adjacent Fabric",
}

Plot spatial variability of seelected variavles

In [ ]:
coef_cols = [
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Počet obyvatel na dům",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Obyvatelstvo - s dlouhodobým pobytem - celkem",
]

n_clusters = len(df_models)

fig, axes = plt.subplots(
    n_clusters, len(coef_cols), figsize=(6 * len(coef_cols), 4 * n_clusters)
)


# global scaling per variable
global_max_dict = {
    col: max(
        row["model"].feature_importances_[col].abs().max()
        for _, row in df_models.iterrows()
    )
    for col in coef_cols
}

global_min_dict = {
    col: max(
        row["model"].feature_importances_[col].abs().min()
        for _, row in df_models.iterrows()
    )
    for col in coef_cols
}

# loop over clusters and variables
for row_i, (_, row) in enumerate(df_models.iterrows()):
    model = row["model"]
    cluster = row["cluster"]

    for col_i, coef_col in enumerate(coef_cols):
        ax = axes[row_i, col_i]

        series = model.feature_importances_[coef_col]
        tmp = data.assign(_coef_tmp=series.values)

        tmp.plot(
            column="_coef_tmp",
            ax=ax,
            cmap="RdYlGn_r",
            vmin=global_min_dict[coef_col],
            vmax=global_max_dict[coef_col],
            legend=False,
            missing_kwds={"color": "lightgray"},
        )

        if row_i == 0:
            ax.set_title(coef_col, fontsize=12)

        if col_i == 0:
            ax.set_ylabel(morpho_names[cluster], fontsize=12)

        ax.axis("off")
plt.tight_layout()
plt.show()

plot sptial variability of f1 macro

In [ ]:
global_f1_max = max(
    row["model"].local_oob_f1_macro_.max() for _, row in df_models.iterrows()
)

global_f1_min = min(
    row["model"].local_oob_f1_macro_.min() for _, row in df_models.iterrows()
)


fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flat

for ax, (_, row) in zip(axes, df_models.iterrows(), strict=False):
    model = row["model"]
    cluster = row["cluster"]

    # extract local F1-macro values
    f1_series = model.local_oob_f1_macro_

    # temporary gdf
    tmp = data.assign(_f1_tmp=f1_series.values)

    tmp.plot(
        ax=ax,
        column="_f1_tmp",
        cmap="YlGnBu",
        legend=True,
        vmin=global_f1_min,
        vmax=global_f1_max,
        missing_kwds={"color": "lightgray"},
        legend_kwds={"shrink": 0.6},
    )

    ax.set_title(morpho_names[cluster])
    ax.set_axis_off()
    ax.axis("off")


plt.tight_layout()
plt.show()

# Composite

In [ ]:
clusters = [1, 2, 3, 4, 5, 7, 8]
reduction = "fa"
model_type = "rf"

# Create a copy of the original data for plotting
plot_data = data.copy()

# Add a column to store the local coefficient values
plot_data["local_coef"] = None

for cluster in clusters:
    # Load model
    model_path = f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib"
    with open(model_path, "rb") as f:
        model = joblib.load(f)

    # Filter original cluster geometries
    compact = data[data["Cluster"] == cluster]

    # Create GeoDataFrame with model results
    compact_model = gpd.GeoDataFrame(
        {
            "geometry": model.geometry,
            "local_coef": model.feature_importances_[
                "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý"
            ],
        }
    )

    # Spatial join to map model values onto original geometries
    compacts = gpd.sjoin(compact_model, compact, predicate="within")
    compacts = compacts.drop(columns="index_right")

    # Assign local_coef values back to original geometries
    plot_data.loc[plot_data["Cluster"] == cluster, "local_coef"] = compacts[
        "local_coef"
    ].values
    plot_data["local_coef"] = plot_data["local_coef"].astype(float)
# Now plot all clusters in one map
fig, ax = plt.subplots(figsize=(12, 10))
plot_data.plot(
    column="local_coef",
    cmap="RdYlGn_r",
    legend=True,
    missing_kwds={"color": "lightgray"},
    legend_kwds={"shrink": 0.6},
    ax=ax,
)
ax.set_title(
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý"
)
ax.set_axis_off()
plt.show()

In [ ]:
reduction = "fa"
model_type = "rf"

# Create a copy of the original data for plotting
plot_data = data.copy()

# Add a column to store f1_macro values
plot_data["f1_macro"] = None

for cluster in [1, 3, 4, 5, 6, 7, 8]:
    # Load model
    model_path = f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib"
    with open(model_path, "rb") as f:
        model = joblib.load(f)

    # Filter original cluster geometries
    compact = data[data["Cluster"] == cluster]

    # Create GeoDataFrame with model results
    compact_model = gpd.GeoDataFrame(
        {
            "geometry": model.geometry,
            "local_pooled_f1_macro": model.local_oob_f1_macro_,
        }
    )

    # Spatial join to map model values onto original geometries
    compacts = gpd.sjoin(compact_model, compact, predicate="within")
    compacts = compacts.drop(columns="index_right")

    # Assign f1_macro values back to original geometries
    plot_data.loc[plot_data["Cluster"] == cluster, "f1_macro"] = compacts[
        "local_pooled_f1_macro"
    ].values

In [ ]:
plot_data["f1_macro"] = plot_data["f1_macro"].astype(float)

In [ ]:
# Now plot all clusters in one map
fig, ax = plt.subplots(figsize=(12, 10))
plot_data.plot(
    column="f1_macro",
    vmin=0.5,
    vmax=0.8,
    cmap="YlGnBu",
    legend=True,
    missing_kwds={"color": "pink"},
    ax=ax,
)
ax.set_title("Local pooled F1 macro across all clusters")
ax.set_axis_off()
plt.show()